# **1. pyannote**
음성에서 화자를 구분하는 화자 분리(Speaker Diarization) 라이브러리임!

### **1-1. 화자 분리**
화자 분리는 긴 녹음 파일을 듣고 언제 누가 말했는지를 시간 구간으로 나누는 작업이다
```
예를 들어
0.0초 ~ 2.4초 까지 실제 발화(사람이 말 한 내용)가 "안녕하세요. 회의를 시작하겠습니다." 라고 했을 때
화자 분리 모델은 SPEAKER_00: 0.0초 ~ 2.4초 이런식으로 나올거임(구간처리)
그리고 또 잠시 있다가 2.8 ~ 5.0초까지 "네, 자료를 준비했습니다" 라고 다른 사람이 말을 하면
SPEAKER_01: 2.8초 ~ 5.0초 이런식으로 나올거임(구간처리)
그리고 다시 첫번째 말했던 사람이 5.8 ~ 7.0초까지 "첫 번째 안건부터 보겠습니다." 라고 말을 하면
SPEAKER_00: 5.8초 ~ 7.0초 이런식으로 나올거임(구간처리)
다시 또 두번째 말했던 사람이 8.0 ~ 10.1초까지 "자료 너무 좋은 것 같습니다." 라고 말을 하면
SPEAKER_01: 8.0초 ~ 10.1초 이런식으로 나올거임(구간처리)

SPEAKER_00: 0.0초 ~ 2.4초
SPEAKER_01: 2.8초 ~ 5.0초
SPEAKER_00: 5.8초 ~ 7.0초
SPEAKER_01: 8.0초 ~ 10.1초
```

> SPEAKER_00, SPEAKER_01은 실명이나 사용자 ID가 아님! 같은 녹음에서 같은 사람이라고 판단한 구간에 붙인 임시 라벨임

- ASR: 음성 인식
- VAD: 음성 활동 탐지(말 소리가 있는가, 없는가). 음성/비음성 구간
- 화자 분리: 화자별 시간 구간
- 화자 식별: 별도의 등록 음성/인증 체계 필요 > 이 보이스는 등록된 OOO인가? (이 사람이 누구인지까지 확인)

### **1-2. 화자 식별 - 컴퓨터가 사람의 목소리를 구별하는 방법**
- 사람도 전화 통화에서 얼굴을 보지 않고 상대를 구별함
- 말투, 발음 습관, 성대와 발성 특성, 음색과 억양 등이 다르기 때문에 구별 가능
- 그래서 AI는 이런 특징을 학습한 신경망으로 음성을 숫자 벡터(화자 임베딩)로 변환함

### 1-2-1. 처리 순서
1. 마이크로 말을 하던 wav파일을 준비하던 오디오를 읽음: wav를 로드해서 샘플값 읽음
    - 보통 16kHz로 샘플링된 오디오로 변환 (16kHz 모노라면 1초에 샘플 16,000개)
2. 음성 구간 분석 및 분할: 짧은 시간 창마다 발화 여부와 여러 화자가 활동하는지 추정
    - 중요한건 무음과 발화 경계 중요함! (말이 겹치는지 말을 안하는지..)
3. 화자 임베딩: 각 발화 구간의 목소리 특성을 고정 길이 숫자 벡터로 변환함
    - 문장의 뜻이 무엇인지 알아내기 보다, 누가 말했는지 구별하는 데 초점을 맞춤(음향적인 특징 학습)
4. 화자 군집화: 비슷한 임베딩을 묶어 동일한 화자로 판단함
    - 서로 다른 구간이 멀리 떨어져있어도 같은 화자로 연결할 수 있음
5. 시간축 재구성: 군집 결과를 원래 발화시간에 다시 매핑 후 SPEAKER_00 과 같은 라벨을 부여함
    - 얘네 같은사람이구나~ 하면 라벨로 묶어 보는거임

<img src="./images/ref/07_01.png" alt="화자식별 처리순서" width="600">

가장 베스트는 말이 겹치지 않는게 젤 좋은데~

실제 상황이 그렇지 않음 > 말 겹치는게 많고, 말 끊는 상황이 많은데

그럼 그럴떈?!

### 1-2-2. 겹쳐 말하는 상황
- 두 사람이 예를 들어 4.0 ~ 4.5초에 동시에 말하면 일반 화자 분리 결과는 두 화자의 구간이 겹칠 수 있음
- 그리고 exclusive_speaker_diarization은 같은 시간에 두 화자가 겹치지 않도록 결과를 조정함(하나의 화자만 배정)
- ASR 전사 결과

> 이것들을 종합하여 이용하면, 겹쳐 말하는 상황에서도 어느 화자가 언제 말했는지 보다 정확하게 추정할 수 있음

exclusive_speaker_diarization > ASR 전사 결과 > 실제 동시 발화를 없앤 것 처럼 복원!!

```
예를 들어
강의실에서 수강생 A와 B가 번갈아 발표를 하는 상황일 때
- 음성 구간 탐지: "지금 누가 말하고 있음"
- 특징 추출: "이 구간은 낮고 거침 음색, 다음 구간은 밝은 음색"
- 군집화: "첫 번째와 세 번쨰 발화가 비슷하니까 SPEAKER_00로 묶음"
- 시간 정보 결합: "A는 0~2초, B는 3~5초, A는 6~8초"
```
> 하지만 목소리 높낮이만으로 판단하진 않음. 한 사람이 소리를 높이거나 속삭여도 같은 화자일 수 있고, 다른 두 사라므이 목소리가 비슷할 수 있음. 그래서!! 임베딩은 여러 음향 특징을 함께 학습해야함.

```
근데 그러다 보면 오류가 가장 많이 생길 수 있는 부분이 있는데
그것은 바로 짧게 말하는 구간 + 웃는거 + 겹치는 부분 + 배경음악이 있는경우
이때 모델이 헷갈려할 수 있음
+ 일반적으로 말하다 마이크를 통해 말해도 헷갈려할 수 있고
+ 거리에 따라(멀리서 말하면 음성이 작게 들어옴) 모델이 헷갈려할 수 있음
```

### 1-2-3. 이제 화자분리 모델 써보자

우리 화자분리 모델 써볼건데 허깅페이스에서 가져와서 써볼거임
https://huggingface.co/pyannote/speaker-diarization-community-1

얘는 다운만 받으면 안돼고 허깅페이스 토큰도 있어야함(돈드는거 아님)
https://huggingface.co/settings/tokens

env파일에 HF_TOKEN= 추가함

여기 깃에사 wav파일 받아서 사용
https://github.com/joelcpy/pyannote/blob/main/meeting.wav?utm_source=chatgpt.com

In [9]:
import os
import random
import torch
import soundfile as sf
import librosa
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
# pip install pyannote.audio
from pyannote.audio import Pipeline

In [10]:
AUDIO_PATH = Path("meeting.wav")
PREPARED_PATH = Path("meeting_16k_mono.wav")
TARGET_SR = 16000

# wave: 실제 오디오가 저장된 넘파이 배열
# original_sr: 원본 오디오의 샘플링 레이트
# 샘플링 레이트란? 1초 동안 오디오를 몇 번 샘플링했는지를 나타내는 값 (단위: Hz)
wave, original_sr = sf.read(AUDIO_PATH, dtype="float32")
print("원본 샘플링 레이트:", original_sr)
print("원본 shape:", wave.shape)

원본 샘플링 레이트: 16000
원본 shape: (480000,)


In [11]:
if wave.ndim == 2:
    wave = wave.mean(axis=1) # 스테레오를 모노로 변환

if original_sr != TARGET_SR:
    # 샘플링 레이트가 다르면 리샘플링
    wave = librosa.resample(wave, orig_sr=original_sr, target_sr=TARGET_SR)

sf.write(PREPARED_PATH, wave, TARGET_SR)

print(f"변환 완료: {PREPARED_PATH} / {len(wave) / TARGET_SR:.2f}초")

변환 완료: meeting_16k_mono.wav / 30.00초


In [12]:
load_dotenv()

True

In [13]:
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN ERROR!")

In [14]:
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.xpu.is_available():
    DEVICE = torch.device('xpu')
else:
    DEVICE = torch.device('cpu')

print(DEVICE)

cpu


In [15]:
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-community-1", token=HF_TOKEN)
pipeline.to(DEVICE)

In [16]:
# *** 이거 실행 안되는거 맞음
output = pipeline(str(PREPARED_PATH)) # 정확히 몇 명인지 알고있을 때 num_speakers=인원수 설정하면 좋음
for segment, speaker in output.speakers_diarization:
    print(f"{segment.start:6.2f} - {segment.end:6.2f} | {speaker}")

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, 8, and 9, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.15.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 9:
Traceback (most recent ca

In [17]:
# 오디오 파일 읽기
wave, sr = sf.read(PREPARED_PATH, dtype="float32")

# NumPy → PyTorch Tensor
waveform = torch.from_numpy(wave)

# (샘플 수,) → (1, 샘플 수)
waveform = waveform.unsqueeze(0)

# 오디오 데이터 직접 전달
audio = {
    "waveform": waveform,
    "sample_rate": sr
}

# 화자 분리 실행
output = pipeline(audio)

for segment, speaker in output.speaker_diarization:
    print(f"{segment.start:6.2f} ~ {segment.end:6.2f} | {speaker}")

d:\OSA\14_Agent\qwen_asr_test\.venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:102: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1927.)
  std = sequences.std(dim=-1, correction=1)


  6.73 ~   6.75 | SPEAKER_00
  6.75 ~   7.03 | SPEAKER_01
  7.03 ~   7.19 | SPEAKER_00
  7.59 ~   7.61 | SPEAKER_00
  7.61 ~   8.32 | SPEAKER_01
  8.32 ~   9.92 | SPEAKER_00
  9.92 ~  10.98 | SPEAKER_01
 10.46 ~  14.75 | SPEAKER_00
 14.31 ~  17.88 | SPEAKER_01
 18.02 ~  21.51 | SPEAKER_00
 18.15 ~  18.44 | SPEAKER_01
 21.77 ~  28.50 | SPEAKER_01
 27.86 ~  29.97 | SPEAKER_00


### 1-2-4. 일반 화자 분리 vs Exclusive 화자 분리
- 이 둘의 차이는 동시에 말하는 구간을 어떻게 표현하는지가 다름
- 두 결과 모두 누가 언제 말했는지는 알려주지만, 여러 화자가 동시에 말하는 구간은 어떻게 표현하는지에 차이가 있음
<br><br><br>

[상황: A: 0초부터 5초까지 발화 | B: 3초부터 7초까지 발화 = 3~5초 구간에서 A와 B가 동시에 발화]
- 일반 화자 분리: 실제 대화에서 여러 화자가 동시에 발화하는 구간을 표현할 수 있음
    - {'start': 0, 'end': 5, 'speaker': 'A'}, {'start': 3, 'end': 7, 'speaker': 'B'}
    - = 3~5초 구간에서 A와 B가 동시에 발화
- Exclusive 화자 분리: 하나의 시간 구간에 최대 한 명의 화자만 배정하도록 만든 결과임
    - A: 0초부터 5초까지 발화 | B: 3초부터 7초까지 발화 = 3~5초 구간에서 A와 B가 동시에 말했지만, Exclusive 분리에서는 겹치지 않게 나눔
    - {'start': 0, 'end': 3, 'speaker': 'A'}, {'start': 3, 'end': 5, 'speaker': 'A'}, {'start': 3, 'end': 5, 'speaker': 'B'}, {'start': 5, 'end': 7, 'speaker': 'B'}
    - = 3~5초 구간에서 A와 B가 동시에 발화하지만, 겹치지 않게 나눔 (중첩 발화 정보 손실)

![화자 분리 차이](./images/ref/07_02.png)

In [18]:
def to_rows(annotation):
    return [
        {"start": seg.start, "end": seg.end, "speaker": speaker} for seg, speaker in annotation
    ]

In [19]:
regular_rows = to_rows(output.speaker_diarization)
exclusive_rows = to_rows(output.exclusive_speaker_diarization)

print("일반 분리:", regular_rows[:10])
print("Exclusive 분리:", exclusive_rows[:10])

일반 분리: [{'start': 6.730343750000001, 'end': 6.74721875, 'speaker': 'SPEAKER_00'}, {'start': 6.74721875, 'end': 7.03409375, 'speaker': 'SPEAKER_01'}, {'start': 7.03409375, 'end': 7.185968750000001, 'speaker': 'SPEAKER_00'}, {'start': 7.59096875, 'end': 7.607843750000001, 'speaker': 'SPEAKER_00'}, {'start': 7.607843750000001, 'end': 8.316593750000003, 'speaker': 'SPEAKER_01'}, {'start': 8.316593750000003, 'end': 9.919718750000001, 'speaker': 'SPEAKER_00'}, {'start': 9.919718750000001, 'end': 10.98284375, 'speaker': 'SPEAKER_01'}, {'start': 10.45971875, 'end': 14.745968750000003, 'speaker': 'SPEAKER_00'}, {'start': 14.30721875, 'end': 17.88471875, 'speaker': 'SPEAKER_01'}, {'start': 18.01971875, 'end': 21.512843750000002, 'speaker': 'SPEAKER_00'}]
Exclusive 분리: [{'start': 6.730343750000001, 'end': 6.74721875, 'speaker': 'SPEAKER_00'}, {'start': 6.74721875, 'end': 7.03409375, 'speaker': 'SPEAKER_01'}, {'start': 7.03409375, 'end': 7.185968750000001, 'speaker': 'SPEAKER_00'}, {'start': 7.590

___
# **2. 참가자별 마이크 등록 회의록**

In [20]:
import sounddevice as sd
from pyannote.audio import Model, Inference
from qwen_asr import Qwen3ASRModel

SR = 16000
MIN_RMS = 0.00001

In [ ]:
# pip install omegaconf
# https://huggingface.co/pyannote/embedding 여기 가서 접근권한 받아야함

# 사람의 목소리를 숫자 벡터로 변환하여 서로 같은 사람인지 비교할 수 있도록 해줌!
embedding_model = Model.from_pretrained("pyannote/embedding", token=HF_TOKEN)
embedding_model.to(DEVICE)

# 불러온 임베딩 모델을 실제 음성 데이터에 적용하기 위한 추론 객체 생성
# window="whole": 음성을 어떤 단위로 처리할것인지 설정
# whole은 입력으로 전달된 전체 음성 구간을 하나의 단위로 처리
embedder = Inference(embedding_model, window="whole")

ASR_MODEL_ID = "Qwen/Qwen3-ASR-1.7B"
asr_dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if torch.cuda.is_available() else torch.float32

# 음성에 포함된 말을 텍스트로 변환해줌
# max_inference_batch_size: 최대 음성 입력 배치 크기
# max_new_tokens: 출력할 수 있는 음성 토큰의 최대 개수
asr_model = Qwen3ASRModel.from_pretrained(
    ASR_MODEL_ID, dtype=asr_dtype, device_map="cuda" if torch.cuda.is_available() else "cpu", max_inference_batch_size=1, max_new_tokens=256
)

d:\OSA\14_Agent\qwen_asr_test\.venv\Lib\site-packages\lightning\pytorch\utilities\migration\utils.py:197: Redirecting import of pytorch_lightning.callbacks.early_stopping.EarlyStopping to lightning.pytorch.callbacks.early_stopping.EarlyStopping
d:\OSA\14_Agent\qwen_asr_test\.venv\Lib\site-packages\lightning\pytorch\utilities\migration\utils.py:197: Redirecting import of pytorch_lightning.callbacks.model_checkpoint.ModelCheckpoint to lightning.pytorch.callbacks.model_checkpoint.ModelCheckpoint
d:\OSA\14_Agent\qwen_asr_test\.venv\Lib\site-packages\lightning\pytorch\utilities\migration\migration.py:208: You have multiple `ModelCheckpoint` callback states in this checkpoint, but we found state keys that would end up colliding with each other after an upgrade, which means we can't differentiate which of your checkpoint callbacks needs which states. At least one of your `ModelCheckpoint` callbacks will not be able to reload the state.
Lightning automatically upgraded your loaded checkpoint f

### **2-1. 화자 등록**

In [22]:
# [한 명씩 이름과 목소리를 입력하면, 특징을 뽑아 임베딩을 만드는 함수]

# 마이크로 목소리 녹음
def record_voice(seconds=12):
    print(f"{seconds}초 녹음 중... 자연스럽게 말하세요")
    recording = sd.rec(int(SR * seconds), samplerate=SR, channels=1, dtype="float32", device=1)
    sd.wait()
    return recording[:, 0].copy()

# 목소리를 숫자 벡터로 변환하고 크기를 일정하게 맞춤(정규화)
def embedding_of(wave):
    wave = np.asanyarray(wave, dtype=np.float32).reshape(-1)
    if len(wave) < SR:
        raise ValueError("녹음된 음성이 너무 짧습니다. 화자 식별을 위해 최소 1초 이상의 음성이 필요합니다.")
    tensor = torch.from_numpy(wave).unsqueeze(0)
    with torch.inference_mode():
        vector = np.asarray(embedder({"waveform": tensor, "sample_rate": SR})).reshape(-1)
        norm = np.linalg.norm(vector)
        if norm < 1e-10:
            raise ValueError("유효한 화자 특징을 추출하지 못했습니다.")
        return vector / norm

# 화자별 목소리 특징을 저장할 공간(화자 임베딩 벡터)
registered = {}

# 이름과 목소리 특징을 등록
def enroll(name, seconds=12):
    name = name.strip()
    if not name:
        raise ValueError("이름을 입력하세요.")
    audio = record_voice(seconds)
    rms = float(np.sqrt(np.mean(audio.astype(np.float64) ** 2)))
    print(f"현재 RMS: {rms:.6f} / 최소 기준: {MIN_RMS:.6f}")
    if rms < MIN_RMS:
        raise ValueError("녹음된 음성의 볼륨이 너무 작습니다. 다시 등록하세요.")
    registered[name] = embedding_of(audio)
    print(f"등록 완료: {name} (RMS={rms:.4f})")

In [23]:
enroll("김사과")

12초 녹음 중... 자연스럽게 말하세요
현재 RMS: 0.000355 / 최소 기준: 0.000010
등록 완료: 김사과 (RMS=0.0004)


In [24]:
enroll("오렌지")

12초 녹음 중... 자연스럽게 말하세요
현재 RMS: 0.002303 / 최소 기준: 0.000010
등록 완료: 오렌지 (RMS=0.0023)


In [25]:
print("현재 등록된 참석자:", list(registered))

현재 등록된 참석자: ['김사과', '오렌지']


### 2-1-1. 출력 예시
이 실습 이제 아래에서 할거임
```
[11:49:01] 김사과: 오늘 회의를 시작하겠습니다.
[11:49:11] 오렌지: 네, 첫 번째 안건을 말씀드리겠습니다.
[11:50:03] 김사과: 예산안을 다시 검토해보세요.
```

### **2-2. 화자 분리와 화자 식별**
- 화자 분리: 녹음에서 '언제 누가 말했는지'를 시간 구간으로 나누는 작업
- 컴퓨터는 말투, 발성, 음색을 신경망으로 고정 길이 숫자 벡터(화자 임베딩)로 바꿈 > 문장 뜻보다 누구인지 가르는 특징을 담음
    - 오디오를 읽고 16Hz 모노로 맞춤
    - 짧은 창마다 발화인지, 여러 사람이 겹치는지 봄(무음과 발화의 경계가 중요 > 이걸 음성 활동 탐지 VAD라 함)
    - 발화 구간마다 임베딩을 만듦
    - 비슷한 임베딩을 묶어서 같은 화자로 보게 함
    - 묶음 결과를 시간축에 다시 올려 SPEAKER_00 같은 라벨을 붙여줌
- 일반 화자 분리는 두 사람이 동시에 말한 겹침을 그대로 둘 수 있음. Exclusive 화자 분리는 한 시각에 화자를 한 명만 남김(ASR에 넘기기 쉽지만, 실제 겹침이 사라진 것 처럼 보임)
- 화자 식별: '이 목소리가 등록된 누구인가?' 라는 참가자 음성 임베딩을 저장하고 비교함

In [2]:
# 여기에 import 필요한거 모아모아했음
import os
import random
import numpy as np
import sounddevice as sd
import torch
from dotenv import load_dotenv
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command
from qwen_asr import Qwen3ASRModel
from pyannote.audio import Inference, Model, Pipeline

In [3]:
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.xpu.is_available():
    DEVICE = torch.device('xpu')
else:
    DEVICE = torch.device('cpu')

print(DEVICE)

cpu


📌 이 아래부터 실습 내용은 7_pyannote_1125실습.ipynb 에 진행함!